# Step 10: Generate Next-Best-Action Recommendations

The final step converts customer priority segments into practical commercial actions.

The model already produced a predicted response probability and assigned each customer to a commercial priority segment. This step translates those segments into next-best-action recommendations.

| Segment | Recommended Action |
|---|---|
| High Priority | Priority sales outreach |
| Medium Priority | Targeted marketing follow-up |
| Low Priority | Low-cost nurture campaign |
| Deprioritized | Exclude from current campaign |

This makes the machine learning output directly useful for campaign planning, sales prioritization, and marketing resource allocation.

In [25]:
import pandas as pd
from pathlib import Path

# Load the customer priority segment file created in Step 9
possible_files = [
    Path("../data/processed/customer_priority_segments.csv"),
    Path("../data/processed/commercial_priority_segments.csv"),
    Path("../data/processed/customer_propensity_scores.csv"),
    Path("../data/processed/scored_customers.csv")
]

data_path = None

for file in possible_files:
    if file.exists():
        data_path = file
        break

if data_path is None:
    raise FileNotFoundError(
        "No Step 9 customer segment file was found. Please run Step 9 first."
    )

df = pd.read_csv(data_path)

print(f"Loaded file: {data_path}")
df.head()

Loaded file: ..\data\processed\customer_priority_segments.csv


,customer_id,propensity_score,actual_response,priority_segment,business_meaning,outreach_rank
0,8062,0.999987,1,High Priority,Strong candidate for immediate outreach,1
1,128,0.999987,1,High Priority,Strong candidate for immediate outreach,2
2,2838,0.999986,1,High Priority,Strong candidate for immediate outreach,3
3,2611,0.999985,1,High Priority,Strong candidate for immediate outreach,4
4,1127,0.999982,1,High Priority,Strong candidate for immediate outreach,5


In [26]:
# Create Customer ID if it does not already exist
if "Customer ID" not in df.columns and "customer_id" not in df.columns:
    df["Customer ID"] = range(10001, 10001 + len(df))

# Standardize Customer ID column
if "customer_id" in df.columns:
    df = df.rename(columns={"customer_id": "Customer ID"})

# Standardize probability column
probability_columns = [
    "Probability",
    "probability",
    "propensity_score",
    "Propensity Score",
    "response_probability",
    "predicted_probability"
]

probability_column = None

for col in probability_columns:
    if col in df.columns:
        probability_column = col
        break

if probability_column is None:
    raise ValueError("No probability column was found.")

df = df.rename(columns={probability_column: "Probability"})

# Standardize segment column
segment_columns = [
    "Segment",
    "segment",
    "priority_segment",
    "commercial_priority_segment",
    "propensity_segment"
]

segment_column = None

for col in segment_columns:
    if col in df.columns:
        segment_column = col
        break

if segment_column is None:
    raise ValueError("No segment column was found.")

df = df.rename(columns={segment_column: "Segment"})

df[["Customer ID", "Probability", "Segment"]].head()

,Customer ID,Probability,Segment
0,8062,0.999987,High Priority
1,128,0.999987,High Priority
2,2838,0.999986,High Priority
3,2611,0.999985,High Priority
4,1127,0.999982,High Priority


In [27]:
# Define next-best-action rules
next_best_action_rules = {
    "High Priority": "Priority sales outreach",
    "Medium Priority": "Targeted marketing follow-up",
    "Low Priority": "Low-cost nurture campaign",
    "Deprioritized": "Exclude from current campaign"
}

# Apply next-best-action recommendations
df["Next Best Action"] = df["Segment"].map(next_best_action_rules)

# Create final recommendation table
recommendation_df = df[
    ["Customer ID", "Probability", "Segment", "Next Best Action"]
].copy()

# Sort customers from highest to lowest response probability
recommendation_df = recommendation_df.sort_values(
    by="Probability",
    ascending=False
)

# Round probability for business readability
recommendation_df["Probability"] = recommendation_df["Probability"].round(2)

recommendation_df.head(10)

,Customer ID,Probability,Segment,Next Best Action
0,8062,1.0,High Priority,Priority sales outreach
1,128,1.0,High Priority,Priority sales outreach
2,2838,1.0,High Priority,Priority sales outreach
3,2611,1.0,High Priority,Priority sales outreach
4,1127,1.0,High Priority,Priority sales outreach
5,1750,1.0,High Priority,Priority sales outreach
6,2058,1.0,High Priority,Priority sales outreach
7,1094,1.0,High Priority,Priority sales outreach
8,6283,1.0,High Priority,Priority sales outreach
9,7431,1.0,High Priority,Priority sales outreach


In [28]:
# Save final next-best-action recommendation output
output_path = Path("../data/processed/next_best_action_recommendations.csv")

recommendation_df.to_csv(output_path, index=False)

print(f"Next-best-action recommendations saved to: {output_path}")

Next-best-action recommendations saved to: ..\data\processed\next_best_action_recommendations.csv


# Step 11: Business Constraint and Selected-Customer List

In a real commercial campaign, the business cannot contact every customer. Outreach teams have limited sales capacity, marketing budget, and customer-contact limits.

For this project, the business constraint is:

**The campaign team can contact only the top 5,000 customers.**

The final campaign-selection list is created using the following rules:

1. Select customers with the highest predicted response probability.
2. Exclude customers with excessive prior campaign contacts.
3. Exclude customers with a contact-fatigue flag.
4. Prioritize customers with higher expected business value.

This step converts the machine learning output into a practical commercial decision-support tool.

In [29]:
#Cell 1: Define Business Constraints
# Step 11: Add Business Constraints

CAMPAIGN_CAPACITY = 5000
MAX_CURRENT_CAMPAIGN_CONTACTS = 3
MAX_PREVIOUS_CAMPAIGN_CONTACTS = 2

print("Campaign capacity:", CAMPAIGN_CAPACITY)
print("Maximum current campaign contacts allowed:", MAX_CURRENT_CAMPAIGN_CONTACTS)
print("Maximum previous campaign contacts allowed:", MAX_PREVIOUS_CAMPAIGN_CONTACTS)

Campaign capacity: 5000
Maximum current campaign contacts allowed: 3
Maximum previous campaign contacts allowed: 2


In [ ]:
# 
# Step 11 - 
# Cell 2: Create Constraint Flags

import pandas as pd
import os

# Use the correct Step 10 recommendation dataframe
campaign_candidates = recommendation_df.copy()

# Load the original raw dataset directly
raw_data_path = "../data/raw/bank-additional-full.csv"

if not os.path.exists(raw_data_path):
    raise FileNotFoundError("Raw dataset not found at ../data/raw/bank-additional-full.csv")

raw_data = pd.read_csv(raw_data_path, sep=";")

# Create Customer ID in the raw dataset based on row index
raw_data["Customer ID"] = raw_data.index

# Add campaign and previous columns if they are missing
if "campaign" not in campaign_candidates.columns:
    campaign_candidates = campaign_candidates.merge(
        raw_data[["Customer ID", "campaign"]],
        on="Customer ID",
        how="left"
    )

if "previous" not in campaign_candidates.columns:
    campaign_candidates = campaign_candidates.merge(
        raw_data[["Customer ID", "previous"]],
        on="Customer ID",
        how="left"
    )

# Check required columns
required_columns = [
    "Customer ID",
    "Probability",
    "Segment",
    "Next Best Action",
    "campaign",
    "previous"
]

missing_columns = [
    col for col in required_columns
    if col not in campaign_candidates.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

print("Campaign candidates shape:", campaign_candidates.shape)

campaign_candidates.head()

All required columns are available.
Campaign candidates shape: (8238, 6)


,Customer ID,Probability,Segment,Next Best Action,campaign,previous
0,8062,1.0,High Priority,Priority sales outreach,4,0
1,128,1.0,High Priority,Priority sales outreach,2,0
2,2838,1.0,High Priority,Priority sales outreach,1,0
3,2611,1.0,High Priority,Priority sales outreach,3,0
4,1127,1.0,High Priority,Priority sales outreach,5,0


In [33]:
#Cell 3: Apply Business Rules
# Step 11 - Code Cell 3: Apply Business Rules

CAMPAIGN_CAPACITY = 5000
MAX_CURRENT_CAMPAIGN_CONTACTS = 3
MAX_PREVIOUS_CAMPAIGN_CONTACTS = 2

# Create excessive-contact and contact-fatigue flags
campaign_candidates["excessive_prior_contacts"] = (
    campaign_candidates["previous"] > MAX_PREVIOUS_CAMPAIGN_CONTACTS
)

campaign_candidates["contact_fatigue_flag"] = (
    campaign_candidates["campaign"] > MAX_CURRENT_CAMPAIGN_CONTACTS
)

# A customer is eligible only if they do not violate either rule
campaign_candidates["eligible_for_campaign"] = (
    (campaign_candidates["excessive_prior_contacts"] == False) &
    (campaign_candidates["contact_fatigue_flag"] == False)
)

# Display the updated table
campaign_candidates[
    [
        "Customer ID",
        "Probability",
        "Segment",
        "Next Best Action",
        "campaign",
        "previous",
        "excessive_prior_contacts",
        "contact_fatigue_flag",
        "eligible_for_campaign"
    ]
].head(20)


,Customer ID,Probability,Segment,Next Best Action,campaign,previous,excessive_prior_contacts,contact_fatigue_flag,eligible_for_campaign
0,8062,1.0,High Priority,Priority sales outreach,4,0,False,True,False
1,128,1.0,High Priority,Priority sales outreach,2,0,False,False,True
2,2838,1.0,High Priority,Priority sales outreach,1,0,False,False,True
3,2611,1.0,High Priority,Priority sales outreach,3,0,False,False,True
4,1127,1.0,High Priority,Priority sales outreach,5,0,False,True,False
5,1750,1.0,High Priority,Priority sales outreach,1,0,False,False,True
6,2058,1.0,High Priority,Priority sales outreach,3,0,False,False,True
7,1094,1.0,High Priority,Priority sales outreach,8,0,False,True,False
8,6283,1.0,High Priority,Priority sales outreach,10,0,False,True,False
9,7431,1.0,High Priority,Priority sales outreach,1,0,False,False,True


In [34]:

# Step 11 - Cell 4: Create Expected Business Value Score

# Create a simple expected business value proxy
campaign_candidates["business_value_multiplier"] = 1.00

# Customers with fewer prior contacts receive a small value boost
campaign_candidates.loc[
    campaign_candidates["previous"] <= 1,
    "business_value_multiplier"
] += 0.10

# Customers with fewer current campaign contacts receive a small value boost
campaign_candidates.loc[
    campaign_candidates["campaign"] <= 2,
    "business_value_multiplier"
] += 0.10

# Expected business value proxy
campaign_candidates["expected_business_value"] = (
    campaign_candidates["Probability"] * campaign_candidates["business_value_multiplier"]
)

campaign_candidates[
    [
        "Customer ID",
        "Probability",
        "campaign",
        "previous",
        "business_value_multiplier",
        "expected_business_value",
        "Segment",
        "Next Best Action",
        "eligible_for_campaign"
    ]
].head(20)

,Customer ID,Probability,campaign,previous,business_value_multiplier,expected_business_value,Segment,Next Best Action,eligible_for_campaign
0,8062,1.0,4,0,1.1,1.1,High Priority,Priority sales outreach,False
1,128,1.0,2,0,1.2,1.2,High Priority,Priority sales outreach,True
2,2838,1.0,1,0,1.2,1.2,High Priority,Priority sales outreach,True
3,2611,1.0,3,0,1.1,1.1,High Priority,Priority sales outreach,True
4,1127,1.0,5,0,1.1,1.1,High Priority,Priority sales outreach,False
5,1750,1.0,1,0,1.2,1.2,High Priority,Priority sales outreach,True
6,2058,1.0,3,0,1.1,1.1,High Priority,Priority sales outreach,True
7,1094,1.0,8,0,1.1,1.1,High Priority,Priority sales outreach,False
8,6283,1.0,10,0,1.1,1.1,High Priority,Priority sales outreach,False
9,7431,1.0,1,0,1.2,1.2,High Priority,Priority sales outreach,True


In [35]:
# Step 11 - Cell 5: Select the Top 5,000 Customers

# Keep only eligible customers
eligible_customers = campaign_candidates[
    campaign_candidates["eligible_for_campaign"] == True
].copy()

# Rank eligible customers by expected business value first, then probability
selected_customers = (
    eligible_customers
    .sort_values(
        by=["expected_business_value", "Probability"],
        ascending=False
    )
    .head(CAMPAIGN_CAPACITY)
    .reset_index(drop=True)
)

# Add final campaign rank
selected_customers["Campaign Rank"] = selected_customers.index + 1

# Final selected-customer list
selected_customers = selected_customers[
    [
        "Campaign Rank",
        "Customer ID",
        "Probability",
        "Segment",
        "Next Best Action",
        "campaign",
        "previous",
        "contact_fatigue_flag",
        "excessive_prior_contacts",
        "expected_business_value"
    ]
]

selected_customers.head(20)

,Campaign Rank,Customer ID,Probability,Segment,Next Best Action,campaign,previous,contact_fatigue_flag,excessive_prior_contacts,expected_business_value
0,1,128,1.0,High Priority,Priority sales outreach,2,0,False,False,1.2
1,2,2838,1.0,High Priority,Priority sales outreach,1,0,False,False,1.2
2,3,1750,1.0,High Priority,Priority sales outreach,1,0,False,False,1.2
3,4,7431,1.0,High Priority,Priority sales outreach,1,0,False,False,1.2
4,5,3240,1.0,High Priority,Priority sales outreach,1,0,False,False,1.2
5,6,6744,1.0,High Priority,Priority sales outreach,2,0,False,False,1.2
6,7,6734,1.0,High Priority,Priority sales outreach,2,0,False,False,1.2
7,8,1951,1.0,High Priority,Priority sales outreach,2,0,False,False,1.2
8,9,22,1.0,High Priority,Priority sales outreach,1,0,False,False,1.2
9,10,5001,1.0,High Priority,Priority sales outreach,2,0,False,False,1.2


In [36]:
# Step 11 - Cell 6: Save the Selected-Customer List

import os

# Create processed folder if it does not exist
os.makedirs("../data/processed", exist_ok=True)

# Save final selected customer list
selected_customers.to_csv(
    "../data/processed/selected_top_5000_customers.csv",
    index=False
)

print("Selected customer list saved successfully.")
print("Number of selected customers:", selected_customers.shape[0])

Selected customer list saved successfully.
Number of selected customers: 5000


In [37]:
# Step 11 - Code Cell 7: Business Summary

print("Business Constraint Summary")
print("---------------------------")
print("Total scored customers:", campaign_candidates.shape[0])
print("Eligible customers after constraints:", eligible_customers.shape[0])
print("Final selected customers:", selected_customers.shape[0])
print("Campaign capacity:", CAMPAIGN_CAPACITY)

print("\nSelected customer segment distribution:")
print(selected_customers["Segment"].value_counts())

print("\nAverage predicted probability of selected customers:")
print(round(selected_customers["Probability"].mean(), 4))

print("\nAverage expected business value of selected customers:")
print(round(selected_customers["expected_business_value"].mean(), 4))

Business Constraint Summary
---------------------------
Total scored customers: 8238
Eligible customers after constraints: 6773
Final selected customers: 5000
Campaign capacity: 5000

Selected customer segment distribution:
Segment
Deprioritized    4224
High Priority     776
Name: count, dtype: int64

Average predicted probability of selected customers:
0.1552

Average expected business value of selected customers:
0.1837


# Step 12: Model Explainability

## Objective

The objective of this step is to explain which customer, campaign, and prior-contact variables influence campaign response predictions.

Explainability is important because commercial teams should not rely only on model accuracy. They also need to understand why the model ranks some customers higher than others.

This step creates a feature importance output to identify the most influential variables in predicting whether a customer is likely to respond to a campaign.

## Why Explainability Matters

Model explainability supports:

- Business trust
- Campaign transparency
- Model governance
- Better communication with marketing and sales teams
- More informed outreach strategy decisions

Important variables may include:

- Previous campaign outcome
- Contact type
- Customer job category
- Education
- Age
- Number of campaign contacts
- Prior contact history
- Days since previous contact
- Campaign timing

Note: The UCI Bank Marketing Additional dataset does not contain a `balance` variable. Therefore, balance is not included in this project unless a different version of the dataset is used.